In [ ]:
model_pipelines_gscv = {
    "PLSRegression": {
        "model": PLSRegression(),
        "params": {"model__n_components": np.arange(2, 21, 2)}
    },
    "ElasticNet": {
        "model": ElasticNet(max_iter=10000),
        "params": {
            "model__alpha": np.logspace(-4, 1, 6),
            "model__l1_ratio": np.linspace(0.1, 1.0, 5)  # 0.1 to 1.0 balances L1 and L2
        }
    },
    "BayesianRidge": {
        "model": BayesianRidge(),
        "params": {
            "model__alpha_1": np.logspace(-6, -1, 6),
            "model__alpha_2": np.logspace(-6, -1, 6),
            "model__lambda_1": np.logspace(-6, -1, 6),
            "model__lambda_2": np.logspace(-6, -1, 6)
        }
    },
    "SVR": {
        "model": SVR(),
        "params": {
            "model__kernel": ["rbf", "linear"],
            "model__C": np.logspace(-2, 2, 6),
            "model__gamma": ["scale", "auto"]
        }
    },
    "RandomForestRegressor": {
        "model": RandomForestRegressor(random_state=RANDOM_SEED),
        "params": {
            "model__n_estimators": np.arange(100, 301, 50),
            "model__max_depth": [None] + list(np.arange(5, 31, 5)),
            "model__min_samples_split": np.arange(2, 11, 2)
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=RANDOM_SEED),
        "params": {
            "model__n_estimators": list(range(100, 351, 50)),
            "model__max_depth": list(range(3, 11, 2)),
            "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
            "model__subsample": [0.6, 0.8, 1.0]
        }
    }
}

""" 
        if ENABLE_HYPERPARAMETER_TUNING:
            grid_search = GridSearchCV(
                estimator=pipeline,
                param_grid=config["params"],
                cv=splits,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                verbose=1,
                return_train_score=True
            )

            try:
                grid_search.fit(X_train, y_train_transformed)
            except Exception as e:
                logger.warning(f"Training failed for {model_name} on {target}: {e}")
                continue
            
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_

        else:
            try:
                pipeline.fit(X_train, y_train_transformed)
            except Exception as e:
                logger.warning(f"Training failed for {model_name} on {target} (no tuning): {e}")
                continue
            
            best_model = pipeline
            best_params = "Default (no tuning)"
        """

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from sklearn.model_selection import (
    GridSearchCV,
    KFold, 
    GroupKFold, 
    GroupShuffleSplit, 
    cross_validate,
    StratifiedShuffleSplit,
    StratifiedKFold,
    train_test_split
)
from sklearn.linear_model import BayesianRidge, ElasticNet
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import GradientBoostingRegressor
import joblib
from ml import get_training_logger, train_models_for_target, cluster_and_split

# ---------------------------
# Constants and Settings
# ---------------------------
RANDOM_SEED = random.randint(0, 1000000)
ENABLE_HYPERPARAMETER_TUNING = True
USE_BAYES_OPT = True
USE_GROUP_SPLIT = True  # Set to False for simple train-test split
TEST_SIZE = 0.2
SPLIT_STRATEGY = 'kfold' # 'kfold', 'groupkfold' or 'stratifiedshuffle'

columns_to_transform = ['Clay%', 'Coarse element %', 'C.E.C (meq / 100g)', 'Na2O ppm', 'Cl- ppm', 
                        'Organic matter%', 'Exchangeable K2O ppm', 'P2O5 ehangable Olsen ppm',
                        'MgO ppm', 'Iron ppm DTPA', 'Mn ppm DTPA']

os.makedirs("final_models", exist_ok=True)
os.makedirs("metrics", exist_ok=True)

# ---------------------------
# Logging Configuration
# ---------------------------
logger = get_training_logger(name='AlMoutmir Soil Models Training', log_file='metrics/model_training.log')
logger.info("Logger initialized successfully.")
    
# ---------------------------
# GAM Definition
# ---------------------------

# ---------------------------
# Define Columns and Model Pipelines
# ---------------------------
target_columns = ['Sand%', 
                  'Clay%', 
                  'Total_Silt%', 
                  'pH_water', 
                  'CE_ext1 / 5 ms / cm',
                  'C / N ratio', 
                  'Organic matter%', 
                  'Exchangeable K2O ppm', 
                  'P2O5 ehangable Olsen ppm'
                  ]

eliminated = ['uuid', 'Latitude_Y', 'Longitude_X', 'Month', 'Fine silt%', 'Coarse silt%', 'CaCo3%_Total', 'CaO ppm']
feature_columns = [col for col in merged_dataset.columns if col not in target_columns + eliminated]

# ---------------------------
# KMeans Clustering for Spatial Segmentation
# ---------------------------

logger.info("Clustering dataset based on spatial location (KMeans)...")
merged_df = cluster_and_split(merged_dataset, seed=RANDOM_SEED)
logger.info(f"Cluster value counts:\n{merged_df['cluster'].value_counts().to_string()}")

# ---------------------------
# Model Pipelines Configuration
# ---------------------------

num_features = 50

model_pipelines = {
    "PLSRegression": {
        "model": PLSRegression(),
        "params": {
            "model__n_components": np.arange(2, min(21, num_features + 1), 2)
        }
    },
    "ElasticNet": {
        "model": ElasticNet(max_iter=10000),
        "params": {
            "model__alpha": np.logspace(-4, 0.5, 8),
            "model__l1_ratio": np.concatenate([
                np.linspace(0.1, 0.5, 3),
                np.linspace(0.6, 1.0, 5)
            ])
        }
    },
    "BayesianRidge": {
        "model": BayesianRidge(),
        "params": {
            "model__alpha_1": np.logspace(-6, -2, 5),
            "model__alpha_2": np.logspace(-6, -2, 5),
            "model__lambda_1": np.logspace(-6, -2, 5),
            "model__lambda_2": np.logspace(-6, -2, 5)
        }
    },
    "SVR": {
        "model": SVR(),
        "params": {
            "model__kernel": ["rbf", "linear"],
            "model__C": np.logspace(-1, 2, 6),
            "model__gamma": ["scale", "auto"],
            "model__epsilon": [0.01, 0.1, 0.2]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=RANDOM_SEED),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__subsample": [0.6, 0.8, 1.0],
            "model__min_samples_split": [2, 5, 10]
        }
    }
}

# ---------------------------
# Dataset Splitting
# ---------------------------
""" 
logger.info("Splitting dataset into train and test groups using GroupShuffleSplit...")

# Drop only columns in X that are all NaNs, and drop rows with any NaNs in X
valid_feature_columns = merged_df[feature_columns].dropna(axis=1, how='all').columns.tolist()
X_base = merged_df[valid_feature_columns]
X_base = X_base.apply(lambda row: row.fillna(row.mean()), axis=1)
merged_cleaned = merged_df.loc[X_base.index]

groups_all = merged_cleaned['cluster']
y_all = merged_cleaned[target_columns]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X_base, y_all, groups=groups_all))

X_train_all, X_test_all = X_base.iloc[train_idx], X_base.iloc[test_idx]
y_train_all, y_test_all = y_all.iloc[train_idx], y_all.iloc[test_idx]
groups_train_all = groups_all.iloc[train_idx]
groups_test_all = groups_all.iloc[test_idx]

logger.info("Splitting dataset into train/test groups using GroupShuffleSplit... "
            f"Train: {X_base.shape[0]*0.8:.0f} rows | Test: {X_base.shape[0]*0.2:.0f} rows")

logger.info(f"Train group distribution:\n{groups_train_all.value_counts().sort_index().to_string()}")
logger.info(f"Test group distribution:\n{groups_test_all.value_counts().sort_index().to_string()}")
logger.info(f"Test set includes groups: {sorted(groups_test_all.unique())}")
"""

# Drop only columns in X that are all NaNs, and drop rows with any NaNs in X
valid_feature_columns = merged_df[feature_columns].dropna(axis=1, how='all').columns.tolist()
X_base = merged_df[valid_feature_columns]
X_base = X_base.apply(lambda row: row.fillna(row.mean()), axis=1)
merged_cleaned = merged_df.loc[X_base.index]

groups_all = merged_cleaned['cluster']
y_all = merged_cleaned[target_columns]

if USE_GROUP_SPLIT:
    logger.info("Splitting dataset into train and test groups using GroupShuffleSplit...")
    
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
    train_idx, test_idx = next(gss.split(X_base, y_all, groups=groups_all))
    
    X_train_all, X_test_all = X_base.iloc[train_idx], X_base.iloc[test_idx]
    y_train_all, y_test_all = y_all.iloc[train_idx], y_all.iloc[test_idx]
    groups_train_all = groups_all.iloc[train_idx]
    groups_test_all = groups_all.iloc[test_idx]
    
    logger.info("Splitting dataset into train/test groups using GroupShuffleSplit... "
                f"Train: {len(train_idx)} rows | Test: {len(test_idx)} rows")
    
    logger.info(f"Train group distribution:\n{groups_train_all.value_counts().sort_index().to_string()}")
    logger.info(f"Test group distribution:\n{groups_test_all.value_counts().sort_index().to_string()}")
    logger.info(f"Test set includes groups: {sorted(groups_test_all.unique())}")
else:
    logger.info("Splitting dataset using simple train-test split...")
    
    X_train_all, X_test_all, y_train_all, y_test_all, groups_train_all, groups_test_all = train_test_split(
        X_base, y_all, groups_all, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )
    
    logger.info("Splitting dataset using simple train-test split... "
                f"Train: {len(X_train_all)} rows | Test: {len(X_test_all)} rows")
    
    logger.info(f"Train group distribution:\n{groups_train_all.value_counts().sort_index().to_string()}")
    logger.info(f"Test group distribution:\n{groups_test_all.value_counts().sort_index().to_string()}")

# ---------------------------
# Run Training for All Targets
# ---------------------------
logger.info("Starting full training process for all targets...")
all_results = []
for target in target_columns:
    res = train_models_for_target(
            target=target,
            X_train=X_train_all,
            y_train=y_train_all[target],
            X_test=X_test_all,
            y_test=y_test_all[target],
            groups_train=groups_train_all,
            model_pipelines=model_pipelines,
            columns_to_transform=columns_to_transform,
            seed=RANDOM_SEED,
            split_strategy=SPLIT_STRATEGY,
            enable_hyperparameter_tuning=ENABLE_HYPERPARAMETER_TUNING, 
            use_bayes_opt=USE_BAYES_OPT
        )
    if res:
        all_results.extend(res)

# ---------------------------
# Save and Display Final Metrics
# ---------------------------
metrics_df = pd.DataFrame(all_results)
top_models = metrics_df.groupby("target").apply(lambda df: df.sort_values("Test_R2", ascending=False).head(1))
top_models.to_csv("metrics/best_models_summary.csv", index=False)
metrics_df.to_csv("metrics/all_models_metrics.csv", index=False)
logger.info("Metrics saved to metrics/all_models_metrics.csv")
print("\nFinal Metrics DataFrame:")

joblib.dump({
    "X_test": X_test_all,
    "y_test": y_test_all
}, "final_models/test_sets.pkl")
logger.info("Test sets saved to final_models/test_sets.pkl")

# Display the metrics DataFrame
display(metrics_df)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

# Load test data
test_sets = joblib.load("final_models/test_sets.pkl")
X_test = test_sets["X_test"]
y_test_dict = test_sets["y_test"]

# Model directory and transformation setup
model_dir = "final_models"
log_transformer = LogTransformer()  # Ensure this is defined/imported
model_names = list(model_pipelines.keys())  # e.g., {"ridge": ..., "rf": ...}
n_targets = len(target_columns)
n_models = len(model_names)

# Set up plot grid
fig, axes = plt.subplots(n_targets, n_models, figsize=(5 * n_models, 5 * n_targets))
fig.suptitle("Test set Observed vs Predicted", fontsize=18)

# Ensure axes is always 2D
axes = np.atleast_2d(axes)

# Loop over targets and models
for i, target in enumerate(target_columns):
    y_test_full = y_test_dict[target]
    is_log = target in columns_to_transform

    for j, model_name in enumerate(model_names):
        ax = axes[i, j]
        model_file = os.path.join(model_dir, f"{target.replace('/', '_')}_{model_name}.pkl")

        if not os.path.exists(model_file):
            ax.set_title(f"{target} - {model_name} (Missing)")
            ax.axis("off")
            continue

        model = joblib.load(model_file)
        y_pred_raw = model.predict(X_test)

        # Convert to Series for safe alignment
        y_test = pd.Series(y_test_full, index=X_test.index)
        y_pred = pd.Series(y_pred_raw, index=X_test.index)

        # Inverse transform if needed
        if is_log:
            y_pred = log_transformer.inverse_transform(y_pred)

        # Filter out NaNs
        valid_mask = y_test.notna() & y_pred.notna()
        y_test_clean = y_test.loc[valid_mask]
        y_pred_clean = y_pred.loc[valid_mask]

        # Compute metrics
        r2 = r2_score(y_test_clean, y_pred_clean)
        rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred_clean))
        bias = np.mean(y_pred_clean - y_test_clean)

        # Plot
        ax.scatter(y_test_clean, y_pred_clean, alpha=0.5)
        min_val = min(y_test_clean.min(), y_pred_clean.min())
        max_val = max(y_test_clean.max(), y_pred_clean.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1)
        ax.set_title(f"{target} - {model_name}")
        ax.set_xlabel("Observed")
        ax.set_ylabel("Predicted")
        ax.set_aspect('equal', 'box')

        # Annotate with metrics
        ax.text(0.05, 0.95,
                f"R²: {r2:.2f}\nRMSE: {rmse:.2f}\nBias: {bias:.2f}",
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Final layout
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
from matplotlib.colors import TABLEAU_COLORS  # For distinct colors

# Define your transformers and model pipelines (make sure these are defined)
log_transformer = LogTransformer()  # Ensure this is defined/imported

# Find all trial folders
base_dir = "."  # or specify your base directory
trial_folders = [f for f in os.listdir(base_dir) 
                if os.path.isdir(os.path.join(base_dir, f)) 
                and f.startswith("Trial_")]

# Check for random_split folder
random_split_folder = "random_split" if os.path.exists(os.path.join(base_dir, "random_split")) else None

# Get colors for each trial
colors = list(TABLEAU_COLORS.values())[:len(trial_folders)]
trial_color_map = dict(zip(trial_folders, colors))

# Model names
model_names = list(model_pipelines.keys())
n_targets = len(target_columns)
n_models = len(model_names)

# Set up plot grid
fig, axes = plt.subplots(n_targets, n_models, figsize=(5 * n_models, 5 * n_targets))
fig.suptitle("Test set Observed vs Predicted (Multiple Trials vs Random Split)", fontsize=18)

# Ensure axes is always 2D
axes = np.atleast_2d(axes)

# Define style for random split
RANDOM_SPLIT_COLOR = '#444444'  # Dark grey
RANDOM_SPLIT_MARKER = 'X'       # Distinct marker
RANDOM_SPLIT_SIZE = 40          # Slightly larger
RANDOM_SPLIT_ALPHA = 0       # More opaque

# Define style for trials
TRIAL_ALPHA = 0.5               # More transparent
TRIAL_SIZE = 30                 # Standard size

# Loop over targets and models
for i, target in enumerate(target_columns):
    for j, model_name in enumerate(model_names):
        ax = axes[i, j]
        ax.set_title(f"{target} - {model_name}")
        ax.set_xlabel("Observed")
        ax.set_ylabel("Predicted")
        
        # Store metrics for all trials (excluding random_split)
        all_r2 = []
        all_rmse = []
        all_bias = []
        
        # Store all data points for this subplot to set equal limits
        all_y_test = []
        all_y_pred = []
        
        # First plot the trial data in background
        for trial in trial_folders:
            try:
                # Load test data for this trial
                test_sets = joblib.load(os.path.join(trial, "test_sets.pkl"))
                X_test = test_sets["X_test"]
                y_test_dict = test_sets["y_test"]
                y_test_full = y_test_dict[target]
                
                # Load model
                model_file = os.path.join(trial, f"{target.replace('/', '_')}_{model_name}.pkl")
                if not os.path.exists(model_file):
                    continue
                
                model = joblib.load(model_file)
                y_pred_raw = model.predict(X_test)
                
                # Convert to Series for safe alignment
                y_test = pd.Series(y_test_full, index=X_test.index)
                y_pred = pd.Series(y_pred_raw, index=X_test.index)
                
                # Inverse transform if needed
                is_log = target in columns_to_transform
                if is_log:
                    y_pred = log_transformer.inverse_transform(y_pred)
                
                # Filter out NaNs
                valid_mask = y_test.notna() & y_pred.notna()
                y_test_clean = y_test.loc[valid_mask]
                y_pred_clean = y_pred.loc[valid_mask]
                
                # Store data points
                all_y_test.extend(y_test_clean.values)
                all_y_pred.extend(y_pred_clean.values)
                
                # Compute metrics
                r2 = r2_score(y_test_clean, y_pred_clean)
                rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred_clean))
                bias = np.mean(y_pred_clean - y_test_clean)
                
                # Store metrics
                all_r2.append(r2)
                all_rmse.append(rmse)
                all_bias.append(bias)
                
                # Plot trial data with reduced prominence
                ax.scatter(y_test_clean, y_pred_clean, 
                          alpha=TRIAL_ALPHA, s=TRIAL_SIZE,
                          color=trial_color_map[trial], 
                          label=trial)
                
            except Exception as e:
                print(f"Error processing {trial} for {target}-{model_name}: {str(e)}")
                continue
        
        # Then plot random_split data in foreground if it exists
        if random_split_folder:
            try:
                # Load test data for random_split
                test_sets = joblib.load(os.path.join(random_split_folder, "test_sets.pkl"))
                X_test = test_sets["X_test"]
                y_test_dict = test_sets["y_test"]
                y_test_full = y_test_dict[target]
                
                # Load model
                model_file = os.path.join(random_split_folder, f"{target.replace('/', '_')}_{model_name}.pkl")
                if os.path.exists(model_file):
                    model = joblib.load(model_file)
                    y_pred_raw = model.predict(X_test)
                    
                    # Convert to Series for safe alignment
                    y_test = pd.Series(y_test_full, index=X_test.index)
                    y_pred = pd.Series(y_pred_raw, index=X_test.index)
                    
                    # Inverse transform if needed
                    is_log = target in columns_to_transform
                    if is_log:
                        y_pred = log_transformer.inverse_transform(y_pred)
                    
                    # Filter out NaNs
                    valid_mask = y_test.notna() & y_pred.notna()
                    y_test_clean = y_test.loc[valid_mask]
                    y_pred_clean = y_pred.loc[valid_mask]
                    
                    # Plot random split with prominent style
                    ax.scatter(y_test_clean, y_pred_clean, 
                              alpha=RANDOM_SPLIT_ALPHA, s=RANDOM_SPLIT_SIZE,
                              color=RANDOM_SPLIT_COLOR, marker=RANDOM_SPLIT_MARKER,
                              label='Random split', zorder=3)  # zorder brings to front
                    
                    # Store for limits calculation but don't include in metrics
                    all_y_test.extend(y_test_clean.values)
                    all_y_pred.extend(y_pred_clean.values)
                    
            except Exception as e:
                print(f"Error processing random_split for {target}-{model_name}: {str(e)}")
        
        # Add reference line and set aspect ratio if we have any data
        if len(all_y_test) > 0:
            # Get overall min and max values
            min_val = min(all_y_test + all_y_pred)
            max_val = max(all_y_test + all_y_pred)
            
            # Add some padding to the limits
            padding = 0.05 * (max_val - min_val)
            ax.set_xlim(min_val - padding, max_val + padding)
            ax.set_ylim(min_val - padding, max_val + padding)
            
            # Set 1:1 aspect ratio
            ax.set_aspect('equal', 'box')
            
            # Add reference line
            ax.plot([min_val - padding, max_val + padding], 
                    [min_val - padding, max_val + padding], 
                    'r--', lw=1, zorder=2)
            
            # Calculate average metrics (only from main trials)
            if len(all_r2) > 0:
                avg_r2 = np.mean(all_r2)
                avg_rmse = np.mean(all_rmse)
                avg_bias = np.mean(all_bias)
                
                # Annotate with average metrics (excluding random split)
                ax.text(0.05, 0.95,
                        f"Trial Avg R²: {avg_r2:.2f}\nTrial Avg RMSE: {avg_rmse:.2f}\nTrial Avg Bias: {avg_bias:.2f}",
                        transform=ax.transAxes,
                        fontsize=10,
                        verticalalignment='top',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
                        zorder=4)  # Ensure text is on top
            
            # Add legend if we have items to show
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                # Put legend in upper left to avoid covering data
                ax.legend(handles, labels, loc='lower right', fontsize=8)
        else:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.axis('off')

# Final layout
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()